# 01. 기온 라벨 만들기 (열섬 강도)
**작성자: A**

결과물: `data_processed/labels.csv` (열: sensor_id, gu, lat, lon, uhi_day, uhi_night)

In [ ]:
# ── 공통 설정 (모든 노트북 맨 위에 그대로 두기) ──
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AICEE_project'   # 모두 같은 경로 (공유 폴더를 '내 드라이브에 바로가기 추가' 해야 함)
DATA = f'{BASE}/data_processed'
RESULTS = f'{BASE}/results'

import sys
sys.path.append(f'{BASE}/shared')
RANDOM_STATE = 42

USE_DUMMY = True   # ← 진짜 데이터 나오면 False로 바꾸기

## 1) S-DoT 파일 불러오기
`data_raw/sdot/`에 주간 CSV들과 '설치 위치정보.xlsx'를 넣어두기

In [ ]:
import pandas as pd, glob
files = sorted(glob.glob(f'{BASE}/data_raw/sdot/*.csv'))
print(len(files), 'files')
raw = pd.concat([pd.read_csv(f, encoding='utf-8') for f in files], ignore_index=True)  # 글자 깨지면 encoding='cp949'
print(raw.columns.tolist()); raw.head()

## 2) 열 이름 통일
⚠️ 실제 파일의 열 이름을 보고 아래 딕셔너리 왼쪽을 고치기

In [ ]:
COLMAP = {
    '시리얼': 'sensor_id',      # TODO: 실제 열 이름으로
    '측정시간': 'time',          # TODO
    '평균 온도': 'temp',         # TODO
}
d = raw.rename(columns=COLMAP)[['sensor_id', 'time', 'temp']]
d['time'] = pd.to_datetime(d['time'].astype(str), errors='coerce')
d['temp'] = pd.to_numeric(d['temp'], errors='coerce')

## 3) EDA 먼저 (2주차: 모델 전에 데이터부터 읽기)
- 분포 히스토그램, 센서별 결측 비율
- **결측이 왜 생겼나?** (2주차 MCAR/MAR/MNAR) 예: 폭염 때 센서가 꺼진다면 '더운 데이터가 빠지는' 편향 → 보고서에 기록

In [ ]:
print(d['temp'].describe())
miss = d.groupby('sensor_id')['temp'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('결측 많은 센서:'); print(miss.head(10))
d['temp'].hist(bins=60);

## 4) 품질 정리 (보고서 '데이터 품질' 섹션 근거)
- **이상치 vs 극한 사건 구분** (2주차): 폭염 때 실제로 높은 기온은 지우면 안 됨. 물리적으로 불가능한 값, 주변 센서와 따로 노는 센서(고장·직사광선)만 제거
- 제거 기준과 제거한 개수 꼭 기록!

In [ ]:
d = d.dropna()
n0 = len(d)
d = d[(d['temp'] > -10) & (d['temp'] < 50)]          # TODO: 물리적으로 불가능한 값 기준 (팀 합의)
cnt = d.groupby('sensor_id').size()
good = cnt[cnt >= 0.7 * cnt.max()].index              # 기간의 70% 이상 측정된 센서만
d = d[d['sensor_id'].isin(good)]
print(f'제거된 행: {n0 - len(d)}, 남은 센서: {d.sensor_id.nunique()}')

## 5) 열섬 강도 = 센서 기온 − 같은 시각 서울 전체 중앙값

In [ ]:
d['uhi'] = d['temp'] - d.groupby('time')['temp'].transform('median')
d['hour'] = d['time'].dt.hour
DAY_HOURS = [13, 14, 15, 16]      # TODO: 팀 합의
NIGHT_HOURS = [0, 1, 2, 3, 4, 5]  # TODO: 팀 합의
uhi_day = d[d.hour.isin(DAY_HOURS)].groupby('sensor_id')['uhi'].mean().rename('uhi_day')
uhi_night = d[d.hour.isin(NIGHT_HOURS)].groupby('sensor_id')['uhi'].mean().rename('uhi_night')
lab = pd.concat([uhi_day, uhi_night], axis=1).reset_index()
lab.describe()

## 6) 위치(위경도·구) 붙이고 저장

In [ ]:
loc = pd.read_excel(glob.glob(f'{BASE}/data_raw/sdot/*위치*.xlsx')[0])
print(loc.columns.tolist())
LOCMAP = {'시리얼': 'sensor_id', '자치구': 'gu', '위도': 'lat', '경도': 'lon'}   # TODO: 실제 열 이름으로
loc = loc.rename(columns=LOCMAP)[['sensor_id', 'gu', 'lat', 'lon']]
labels = loc.merge(lab, on='sensor_id', how='inner')
labels.to_csv(f'{DATA}/labels.csv', index=False)
print(labels.shape)